# ClimaCity Paris -- Session 5
## Machine Learning avec MLlib, optimisation et bilan d'architecture

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Prérequis** : Avoir complété les Jours 1 et 2 -- la table Delta
`data/output/delta/disponibilite` doit être présente.

---

Ce notebook couvre l'intégralité du Jour 3 du projet ClimaCity Paris.

- **Partie 1 -- Matin (3h30)** : Machine Learning distribué avec MLlib.
  Construction de features, clustering des stations (K-Means), modèle de
  prédiction du taux d'occupation (GBTRegressor), validation croisée,
  et suivi des expériences avec MLflow.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne.  
> Les cellules `# [CORRECTION]` proposent une solution.


---
## Section 0 -- Configuration


In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")


In [5]:
from pathlib import Path
import time, warnings
warnings.filterwarnings("ignore")

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR         = Path("data")
OUTPUT_DIR       = DATA_DIR / "output"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
STATIONS_CSV     = DATA_DIR / "velib" / "stations_info.csv"
MODELS_DIR       = OUTPUT_DIR / "models"
MLFLOW_DIR       = OUTPUT_DIR / "mlruns"

for p in [DELTA_DISPONIBLE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez les Jours 1 et 2"
for p in [MODELS_DIR, MLFLOW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

APP_NAME      = "ClimaCity-Paris-Jour3"
SHUFFLE_PARTS = 8
SEED          = 42

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "8g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- http://localhost:4040")

:: loading settings :: url = jar:file:/Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ms/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ms/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1ce70858-53e9-4a46-9738-ee9a9bcd7a1d;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 70ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.0 from central in [default]
	io.delta#delta-storage;4.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|               

Spark 4.0.4 -- http://localhost:4040


In [7]:
# Chargement de la table Delta consolidée
df = spark.read.format("delta").load(str(DELTA_DISPONIBLE))

# station_id : identifiant entier dérivé du nom de la station (hash positif)
# Le Delta ne contient pas de station_id numérique -- on le recalcule de façon reproductible
df = df.withColumn("station_id", F.abs(F.hash(F.col("nom_station"))))

df.cache()
df.count()   # force le cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()

26/09/21 17:50:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Table consolidée : 5,278,852 lignes  |  19 colonnes
root
 |-- nom_station: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- velos_disponibles: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- station_id: integer (nullable = false)



---
# PARTIE 1 -- Apprentissage automatique avec MLlib

## 1.1 Les trois abstractions fondamentales de MLlib

MLlib repose sur trois interfaces qui permettent de construire des pipelines
ML reproductibles et composables.

### Transformer

Un `Transformer` prend un DataFrame en entrée et retourne un DataFrame enrichi.
Il implémente la méthode `transform(df)`.  
Exemples : `VectorAssembler`, `StandardScaler` (après fit), `Binarizer`.

### Estimator

Un `Estimator` est un algorithme qui s'entraîne sur des données.
Il implémente la méthode `fit(df)` et retourne un `Transformer` (le modèle ajusté).  
Exemples : `GBTRegressor`, `KMeans`, `StandardScaler` (avant fit).

### Pipeline

Un `Pipeline` enchaîne une liste ordonnée de `Transformer` et d'`Estimator`.
Lorsqu'on appelle `pipeline.fit(df)`, chaque étape est exécutée en séquence.
Le résultat est un `PipelineModel`, qui lui-même est un `Transformer`.

```
DataFrame d'entrée
      │
      ▼
  Étape 1 : VectorAssembler  (Transformer)  → features brutes
      │
      ▼
  Étape 2 : StandardScaler   (Estimator)    → calcule mean/std sur train
      │         ↓ fit → StandardScalerModel (Transformer)
      ▼
  Étape 3 : GBTRegressor     (Estimator)    → entraîne le modèle
                ↓ fit → GBTRegressionModel  (Transformer)
      │
      ▼
DataFrame de sortie (avec colonne "prediction")
```

La clé du Pipeline : on appelle `fit()` **une seule fois** sur les données
d'entraînement. Le `PipelineModel` résultant peut être appliqué sur les données
de test **sans risque de fuite d'information** entre train et test.


---
## 1.2 Construction des features

Un bon modèle commence par de bonnes features. Nous allons construire
un vecteur de features à partir des informations disponibles :
contexte temporel, conditions météorologiques, et historique récent de la station.


In [8]:
from pyspark.sql.functions import (
    col, when, sin, cos, lit, lag, lead,
    avg as spark_avg,
    round as spark_round,
)
from pyspark.sql.window import Window

PI = 3.14159265358979

# ── 1. Features temporelles cycliques ────────────────────────────────────────
# L'heure 23 et l'heure 0 sont proches dans le temps mais éloignées en valeur
# brute. On encode l'heure, le jour et le mois sur un cercle (sin/cos).
df_feat = (
    df
    .withColumn("heure_sin", sin(lit(2 * PI) * col("heure") / lit(24)))
    .withColumn("heure_cos", cos(lit(2 * PI) * col("heure") / lit(24)))
    .withColumn("jour_sin",  sin(lit(2 * PI) * col("jour_sem") / lit(7)))
    .withColumn("jour_cos",  cos(lit(2 * PI) * col("jour_sem") / lit(7)))
    .withColumn("mois_sin",  sin(lit(2 * PI) * col("mois") / lit(12)))
    .withColumn("mois_cos",  cos(lit(2 * PI) * col("mois") / lit(12)))
)

# ── 2. Features météo (valeurs manquantes imputées par des constantes) ────────
df_feat = (
    df_feat
    .withColumn("temperature_c",   F.coalesce(col("temperature_c"),   lit(12.0)))
    .withColumn("humidite_pct",     F.coalesce(col("humidite_pct"),    lit(70.0)))
    .withColumn("vent_kmh",         F.coalesce(col("vent_kmh"),        lit(10.0)))
    .withColumn("precipitation_mm", F.coalesce(col("precipitation_mm"), lit(0.0)))
    .withColumn("est_pluie",        F.coalesce(col("est_pluie"), lit(False)).cast("int"))
    .withColumn("is_weekend",       col("is_weekend").cast("int"))
)

# ── 3. Features de lag ────────────────────────────────────────────────────────
# t-1 ≈ 15-20 min avant, t-4 ≈ 60-80 min avant (selon la fréquence des snapshots)
fenetre_station = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
)
fenetre_lag4 = fenetre_station.rowsBetween(-4, -1)

df_feat = (
    df_feat
    .withColumn("taux_t_moins_1", lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_t_moins_4", lag("taux_occupation", 4).over(fenetre_station))
    .withColumn("moy_lag_4",
                spark_round(spark_avg("taux_occupation").over(fenetre_lag4), 4))
)

# ── 4. Variable cible : taux_occupation à t+4 (≈ 1 heure dans le futur) ──────
df_feat = df_feat.withColumn(
    "cible",
    lead("taux_occupation", 4).over(fenetre_station)
)

# ── 5. Suppression des lignes incomplètes (lag/lead génère des nulls) ──────────
FEATURES = [
    "heure_sin", "heure_cos",
    "jour_sin",  "jour_cos",
    "mois_sin",  "mois_cos",
    "is_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie",
    "taux_t_moins_1", "taux_t_moins_4", "moy_lag_4",
    "capacite",
]

df_ml = df_feat.dropna(subset=FEATURES + ["cible"])

print(f"Lignes disponibles pour le ML : {df_ml.count():,}")
print(f"Features ({len(FEATURES)}) : {FEATURES}")

Lignes disponibles pour le ML : 5,267,756
Features (16) : ['heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'taux_t_moins_1', 'taux_t_moins_4', 'moy_lag_4', 'capacite']


## 1.3 Séparation du jeu de données

Créer un jeu de données pour l'entraînement et un jeu de données pour la validation/test

In [9]:
# ── 6. Split train / test ──────────────────────────────────────────────────────
# Split temporel : 2022 = train, 2023 = test
# Un split aléatoire sur des séries temporelles entraîne une fuite d'information :
# le modèle verrait des données du futur pendant l'entraînement.
df_train = df_ml.filter(col("annee") == 2022)
df_test  = df_ml.filter(col("annee") == 2023)

print(f"Train (2022) : {df_train.count():,} lignes")
print(f"Test  (2023) : {df_test.count():,} lignes")
print(f"Ratio        : {df_train.count() / df_ml.count():.1%} / {df_test.count() / df_ml.count():.1%}")

# Mise en cache -- les deux splits seront accédés plusieurs fois
df_train.cache(); df_train.count()
df_test.cache();  df_test.count()

Train (2022) : 3,336,882 lignes
Test  (2023) : 1,930,836 lignes
Ratio        : 63.3% / 36.7%


1930836

## 1.4 Clustering des stations avec K-Means

Avant de prédire la disponibilité individuelle de chaque station, nous allons
**regrouper les stations par profil comportemental** : certaines sont saturées
le matin (quartiers résidentiels qui "envoient" des cyclistes vers le centre),
d'autres le soir, d'autres ont un usage homogène sur la journée.

Ce clustering a deux utilités :
1. Comprendre la géographie fonctionnelle du réseau.
2. Ajouter l'identifiant de cluster comme feature dans le modèle de régression.

### Construction du profil de station

Le vecteur de profil d'une station est son **taux d'occupation moyen à chaque
heure de la journée** -- un vecteur de dimension 24.


In [11]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import ClusteringEvaluator

# ── Profil horaire moyen par station ─────────────────────────────────────────
# Pivot : une ligne par station, une colonne par heure

df_profil = (
    df_ml
    .groupBy("station_id")
    .pivot("heure", list(range(24)))
    .agg(spark_round(spark_avg("taux_occupation"), 4))
)

# Renommage des colonnes pivot (0 -> h00, 1 -> h01, ...)
for h in range(24):
    df_profil = df_profil.withColumnRenamed(str(h), f"h{h:02d}")

colonnes_profil = [f"h{h:02d}" for h in range(24)]
df_profil = df_profil.dropna(subset=colonnes_profil)

print(f"Stations avec profil complet : {df_profil.count()}")
df_profil.show(5)

Stations avec profil complet : 1387
+----------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|station_id|   h00|   h01|   h02|   h03|   h04|   h05|   h06|   h07|   h08|   h09|   h10|   h11|   h12|   h13|   h14|   h15|   h16|   h17|   h18|   h19|   h20|   h21|   h22|   h23|
+----------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|  14003826|0.1285|  0.12| 0.142|0.1329|0.1274|0.1269|0.1192|0.1162|0.1229|0.1182|0.1174|0.1161|0.1062|0.1169|0.1158|0.1176|0.1193|0.1268|0.1314|0.1236|0.1228|0.1256|  0.13|0.1302|
|  15012481| 0.116|0.1117|0.1194|0.1155|0.1151|0.1149|0.1085|0.1002|0.1028|0.1255|0.1515|0.1567| 0.154|0.1375|0.1239|0.1193|0.1173|0.0979|0.0792|0.1039|0.1225|0.1212|0.1189|0.1169|
|  36822892|0.0491|0.0446| 0.052|0.0509|0.0519|0.0521|0.052

In [12]:
# ── Méthode du coude : inertie (WSSSE) en fonction de k ──────────────────────
assembler_profil = VectorAssembler(
    inputCols=colonnes_profil,
    outputCol="features_profil"
)

# StandardScaler : distribution centrée réduite (mean=0, std=1)
scaler_profil = StandardScaler(
    inputCol="features_profil",
    outputCol="features_scaled",
    withStd=True, withMean=True
)

inertie_par_k = {}

for k in range(2, 9):
    kmeans = KMeans(
        featuresCol="features_scaled",
        predictionCol="cluster",
        k=k,
        seed=SEED,
        maxIter=20,
    )
    # Pipeline : assemblage → normalisation → clustering
    pipeline_km = Pipeline(stages=[assembler_profil, scaler_profil, kmeans])
    model_km    = pipeline_km.fit(df_profil)

    # WSSSE : somme des carrés des distances point-centroïde (inertie intra-cluster)
    wssse = model_km.stages[-1].summary.trainingCost
    inertie_par_k[k] = round(wssse, 2)
    print(f"  k={k}  inertie={wssse:,.1f}")

print("\nTableau récapitulatif :")
for k, w in inertie_par_k.items():
    barre = "=" * int(w / max(inertie_par_k.values()) * 40)
    print(f"  k={k}  {w:>12,.1f}  {barre}")

26/09/21 17:55:27 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


  k=2  inertie=9,315.0
  k=3  inertie=6,018.5
  k=4  inertie=4,526.4
  k=5  inertie=3,821.6
  k=6  inertie=3,294.5
  k=7  inertie=2,791.6
  k=8  inertie=2,498.9

Tableau récapitulatif :
  k=2       9,315.0  ========================================
  k=3       6,018.5  =========================
  k=4       4,526.4  ===================
  k=5       3,821.6  ================
  k=6       3,294.5  ==============
  k=7       2,791.6  ===========
  k=8       2,498.9  ==========


In [13]:
# ── Entraînement final avec le k retenu ───────────────────────────────────────
# k=4 est souvent le coude sur ce jeu : profils résidentiel, commercial, touristique, mixte
K_RETENU = 4

kmeans_final = KMeans(
    featuresCol="features_scaled",
    predictionCol="cluster",
    k=K_RETENU,
    seed=SEED,
    maxIter=20,
)
pipeline_km_final = Pipeline(stages=[assembler_profil, scaler_profil, kmeans_final])
model_km_final    = pipeline_km_final.fit(df_profil)

df_clusters = (
    model_km_final
    .transform(df_profil)
    .select("station_id", "cluster")
)

print(f"Répartition des {df_clusters.count()} stations en {K_RETENU} clusters :")
df_clusters.groupBy("cluster").count().orderBy("cluster").show()

Répartition des 1387 stations en 4 clusters :
+-------+-----+
|cluster|count|
+-------+-----+
|      0|  291|
|      1|  587|
|      2|  348|
|      3|  161|
+-------+-----+



In [14]:
# ── Interprétation : profil horaire moyen par cluster ────────────────────────
print("Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :")
(
    df_profil
    .join(df_clusters, on="station_id", how="left")
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("h06"), 3).alias("h06"),
        spark_round(spark_avg("h09"), 3).alias("h09"),
        spark_round(spark_avg("h12"), 3).alias("h12"),
        spark_round(spark_avg("h18"), 3).alias("h18"),
        spark_round(spark_avg("h22"), 3).alias("h22"),
    )
    .orderBy("cluster")
    .show()
)

Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :
+-------+-----+-----+-----+-----+-----+
|cluster|  h06|  h09|  h12|  h18|  h22|
+-------+-----+-----+-----+-----+-----+
|      0|0.393|0.403|0.415|0.428|0.405|
|      1|0.119|0.098|0.087|0.072|0.099|
|      2|0.238| 0.22|0.207|0.196|0.232|
|      3|0.572|0.556|0.561|0.592|0.601|
+-------+-----+-----+-----+-----+-----+



## Folium

Folium s'appuie sur les points forts de l'écosystème Python en termes d'analyse de données et sur ceux de la bibliothèque JS Leaflet our l'affichage cartographique. Manipulez vos données en Python, puis visualisez-les dans une carte via Folium.

### Concepts

Folium facilite la visualisation des données manipulées en Python sur une carte interactive. Il permet à la fois la liaison de données avec une carte géographique pour les visualisations choroplèthes, et de passer comme marqueurs de graphismes riche  vectoriels/binaires/HTML.

La bibliothèque dispose d'un certain nombre de jeux de tuiles intégrés d'OpenStreetMap, Mapbox, etc. Elle prend aussi en charge les jeux de tuiles personnalisés. 

Folium prend en charge les superpositions Image, Video, GeoJSON et TopoJSON et dispose d'un nombre de couches vectorielles intégrées.


In [15]:
# Folium n'est pas dans les dépendances de base -- installation si nécessaire
try:
    import folium
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"], check=True)
    print("folium installé.")

folium installé.


You should consider upgrading via the '/Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/venv/bin/python3 -m pip install --upgrade pip' command.


In [16]:
# ── Visualisation sur carte Folium ─────────────────────────────────────────
import folium
import pandas as pd

PALETTE = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12",
           "#9b59b6", "#1abc9c", "#e67e22"]

# Mapping station_id → nom_station depuis df_ml
df_nom_station  = df_ml.select("station_id", "nom_station").distinct()
df_clusters_nom = df_clusters.join(df_nom_station, on="station_id", how="left")

# Lecture du fichier des stations (pandas) pour récupérer lat/lon
df_stations_pd = pd.read_csv(str(STATIONS_CSV), sep=";")[["name", "lat", "lon"]]

# Jointure et conversion en pandas
df_carte = (
    df_clusters_nom
    .select("nom_station", "cluster")
    .toPandas()
    .merge(df_stations_pd, left_on="nom_station", right_on="name", how="inner")
)
print(f"Stations géolocalisées : {len(df_carte)}")

# Création de la carte
carte = folium.Map(location=[48.8566, 2.3522], zoom_start=12, tiles="CartoDB positron")

for _, ligne in df_carte.iterrows():
    couleur = PALETTE[int(ligne["cluster"]) % len(PALETTE)]
    folium.CircleMarker(
        location=[ligne["lat"], ligne["lon"]],
        radius=5,
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.8,
        popup=folium.Popup(f"Cluster {int(ligne['cluster'])}", parse_html=True),
        tooltip=ligne["nom_station"],
    ).add_to(carte)

# Légende textuelle
legende_html = (
    "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:10px;border:1px solid #ccc;font-size:12px'>"
)
for i in range(K_RETENU):
    legende_html += (
        f"<div><span style='display:inline-block;width:14px;height:14px;"
        f"background:{PALETTE[i]};border-radius:50%;margin-right:6px'></span>"
        f"Cluster {i}</div>"
    )
legende_html += "</div>"
carte.get_root().html.add_child(folium.Element(legende_html))

chemin_carte = OUTPUT_DIR / "carte_clusters_velib.html"
carte.save(str(chemin_carte))
print(f"Carte sauvegardée : {chemin_carte}")
print("Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.")

carte

Stations géolocalisées : 1267
Carte sauvegardée : data/output/carte_clusters_velib.html
Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.


In [17]:
# Ajout du cluster comme feature pour le modèle de régression
df_ml = df_ml.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_train = df_train.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_test  = df_test.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])

FEATURES = FEATURES + ["cluster"]
print(f"Features après ajout du cluster ({len(FEATURES)}) : {FEATURES}")

Features après ajout du cluster (17) : ['heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'taux_t_moins_1', 'taux_t_moins_4', 'moy_lag_4', 'capacite', 'cluster']


---
## 1.5 Modèle de régression : GBTRegressor

Le **Gradient Boosted Tree Regressor** est un algorithme d'ensemble qui construit
séquentiellement des arbres de décision, chacun corrigeant les erreurs du précédent.
C'est l'un des meilleurs algorithmes pour les données tabulaires avec des features
hétérogènes -- exactement notre cas.

MLlib l'implémente de façon distribuée : chaque arbre est construit en parallèle
sur les partitions du DataFrame.


In [18]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# 1. VectorAssembler : colonnes features → vecteur dense "features_brutes"
assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="features_brutes",
    handleInvalid="skip",
)

# 2. StandardScaler : centre et réduit "features_brutes" → "features"
scaler = StandardScaler(
    inputCol="features_brutes",
    outputCol="features",
    withStd=True,
    withMean=True,
)

# 3. GBTRegressor : 50 arbres, pas d'apprentissage 0.1, profondeur 5
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="cible",
    predictionCol="prediction",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    seed=SEED,
)

# Pipeline complet
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

# ── Entraînement ──────────────────────────────────────────────────────────────
print("Entraînement du GBTRegressor (peut prendre 1-2 minutes)...")
t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(df_train)
t_fit = time.perf_counter() - t0
print(f"Entraînement terminé en {t_fit:.1f} s")

Entraînement du GBTRegressor (peut prendre 1-2 minutes)...
Entraînement terminé en 58.1 s


In [19]:
# ── Évaluation sur le test set ────────────────────────────────────────────────
evaluator_rmse = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="mae"
)

df_pred = model_gbt.transform(df_test)

rmse = evaluator_rmse.evaluate(df_pred)
r2   = evaluator_r2.evaluate(df_pred)
mae  = evaluator_mae.evaluate(df_pred)

print(f"Métriques sur le test set (2023) :")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"  MAE  (Mean Absolute Error)     : {mae:.4f}")
print(f"  R²   (coefficient de détermination) : {r2:.4f}")
print()
print("Interprétation :")
print(f"  En moyenne, le modèle se trompe de ±{mae:.3f} sur le taux d'occupation (0-1).")
print(f"  Soit ±{mae*100:.1f} points de pourcentage.")

Métriques sur le test set (2023) :
  RMSE (Root Mean Squared Error) : 0.1011
  MAE  (Mean Absolute Error)     : 0.0600
  R²   (coefficient de détermination) : 0.8045

Interprétation :
  En moyenne, le modèle se trompe de ±0.060 sur le taux d'occupation (0-1).
  Soit ±6.0 points de pourcentage.


In [23]:
# ── Importance des features ────────────────────────────────────────────────────
# Le GBT calcule l'importance de chaque feature (réduction d'impureté moyenne)
model_gbt_stage = model_gbt.stages[-1]   # le GBTRegressionModel
importances     = model_gbt_stage.featureImportances.toArray()

df_importance = (
    pd.DataFrame({
        "feature"   : FEATURES,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Importance des features (top 10) :")
print(f"{'Rang':<5} {'Feature':<25} {'Importance':>12}  Barre")
print("-" * 65)
for i, row in df_importance.head(10).iterrows():
    barre = "█" * int(row["importance"] * 100)
    print(f"  {i+1:<3} {row['feature']:<25} {row['importance']:>12.4f}  {barre}")

Importance des features (top 10) :
Rang  Feature                     Importance  Barre
-----------------------------------------------------------------
  1   taux_t_moins_1                  0.8994  █████████████████████████████████████████████████████████████████████████████████████████
  2   taux_t_moins_4                  0.0241  ██
  3   cluster                         0.0221  ██
  4   heure_cos                       0.0138  █
  5   capacite                        0.0135  █
  6   heure_sin                       0.0122  █
  7   moy_lag_4                       0.0076  
  8   jour_sin                        0.0028  
  9   temperature_c                   0.0021  
  10  vent_kmh                        0.0006  


In [24]:
# ── Analyse des erreurs : où le modèle se trompe-t-il le plus ? ───────────────
df_erreurs = (
    df_pred
    .withColumn("erreur_abs", F.abs(col("prediction") - col("cible")))
    .withColumn("erreur_rel",
        F.abs(col("prediction") - col("cible")) / (col("cible") + F.lit(0.01))
    )
)

print("Erreur absolue moyenne par heure de la journée :")
(
    df_erreurs
    .groupBy("heure")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_horaire"),
        F.count("*").alias("n")
    )
    .orderBy("heure")
    .show(24)
)

Erreur absolue moyenne par heure de la journée :
+-----+-----------+------+
|heure|mae_horaire|     n|
+-----+-----------+------+
|    0|      0.031| 87694|
|    1|     0.0315| 43152|
|    2|     0.0288| 27839|
|    3|     0.0284| 70991|
|    4|     0.0308| 77951|
|    5|     0.0434| 69600|
|    6|      0.062| 90480|
|    7|     0.0846| 76560|
|    8|     0.0889| 93264|
|    9|     0.0781| 86304|
|   10|     0.0673| 79344|
|   11|     0.0637| 91872|
|   12|     0.0637|105792|
|   13|     0.0643| 80736|
|   14|     0.0603| 76560|
|   15|     0.0749|116928|
|   16|     0.0946|101750|
|   17|     0.1043| 52896|
|   18|     0.1002| 50118|
|   19|     0.0794| 65424|
|   20|     0.0513| 86304|
|   21|      0.036|105792|
|   22|     0.0318|104399|
|   23|     0.0328| 89086|
+-----+-----------+------+



In [25]:
# [CORRECTION]
print("MAE par cluster de station :")
(
    df_erreurs
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"),
        spark_round(spark_avg("erreur_rel"), 4).alias("mare_cluster"),
        F.count("*").alias("n_predictions")
    )
    .orderBy(F.desc("mae_cluster"))
    .show()
)
# Les stations de cluster à fort usage (très vides le matin, très pleines le soir)
# sont généralement plus difficiles à prédire car leur dynamique est plus abrupte.


MAE par cluster de station :
+-------+-----------+------------+-------------+
|cluster|mae_cluster|mare_cluster|n_predictions|
+-------+-----------+------------+-------------+
|      0|     0.0808|      0.5994|       403636|
|      3|     0.0727|      0.2597|       223324|
|      2|     0.0651|      1.0161|       486629|
|      1|     0.0431|      1.2115|       817247|
+-------+-----------+------------+-------------+



---
## 1.6 Validation croisée et optimisation des hyperparamètres

Nos hyperparamètres actuels (`maxIter=50`, `maxDepth=5`, `stepSize=0.1`)
ont été choisis arbitrairement. Le `CrossValidator` de MLlib permet
d'évaluer systématiquement plusieurs combinaisons et de choisir la meilleure.

> **Attention** : la validation croisée sur des séries temporelles est
> délicate. Le `CrossValidator` de MLlib effectue un K-Fold standard
> (mélanges aléatoires des données), ce qui peut créer de la fuite
> d'information temporelle. Pour un usage production, on préférerait
> une validation en fenêtre glissante (*time series split*).
> Dans le contexte de ce cours, le K-Fold est acceptable pour comprendre
> le mécanisme.


In [26]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Grille d'hyperparamètres ──────────────────────────────────────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 5])
    .addGrid(gbt.maxIter,   [30, 50])
    .addGrid(gbt.stepSize,  [0.05, 0.1])
    .build()
)
print(f"Combinaisons à évaluer : {len(param_grid)}")

# ── CrossValidator ────────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    seed=SEED,
    parallelism=2    # évalue 2 combinaisons en parallèle
)

# On entraîne sur un sous-échantillon pour limiter le temps de calcul en cours
df_train_sample = df_train.sample(fraction=0.3, seed=SEED).cache()
df_train_sample.count()
print(f"Sous-échantillon train : {df_train_sample.count():,} lignes")

print("\nValidation croisée en cours (peut prendre 3-5 minutes)...")
t0 = time.perf_counter()
cv_model = cv.fit(df_train_sample)
t_cv = time.perf_counter() - t0
print(f"Terminé en {t_cv:.1f} s")

Combinaisons à évaluer : 8
Sous-échantillon train : 1,001,829 lignes

Validation croisée en cours (peut prendre 3-5 minutes)...


26/09/21 18:10:18 WARN BlockManager: Asked to remove block rdd_18111_6, which does not exist
26/09/21 18:11:01 WARN BlockManager: Asked to remove block rdd_21666_7, which does not exist
26/09/21 18:11:01 WARN BlockManager: Asked to remove block rdd_21666_0, which does not exist
26/09/21 18:11:01 WARN BlockManager: Asked to remove block rdd_21666_2, which does not exist
26/09/21 18:11:01 WARN BlockManager: Asked to remove block rdd_21666_1, which does not exist
26/09/21 18:11:03 WARN BlockManager: Asked to remove block rdd_21886_4, which does not exist
26/09/21 18:11:03 WARN BlockManager: Asked to remove block rdd_21886_6, which does not exist


Terminé en 248.0 s


In [29]:
# ── Résultats de la grille ────────────────────────────────────────────────────
avg_metrics = cv_model.avgMetrics
params_list = [
    {p.name: v for p, v in zip(param_grid[0].keys(), combo.values())}
    for combo in param_grid
]

resultats_cv = pd.DataFrame(params_list)
resultats_cv["rmse_cv"] = avg_metrics
resultats_cv = resultats_cv.sort_values("rmse_cv").reset_index(drop=True)

print("Résultats du CrossValidator (triés par RMSE croissant) :")
print(resultats_cv.to_string(index=False))

# Meilleurs hyperparamètres
meilleur = resultats_cv.iloc[0]
print(f"\nMeilleure combinaison :")
for col_name in ["maxDepth", "maxIter", "stepSize", "rmse_cv"]:
    print(f"  {col_name:<12} : {meilleur[col_name]}")

Résultats du CrossValidator (triés par RMSE croissant) :
 maxDepth  maxIter  stepSize  rmse_cv
        5       50      0.10 0.074087
        5       30      0.10 0.074609
        5       50      0.05 0.074741
        5       30      0.05 0.075297
        3       50      0.10 0.076265
        3       30      0.10 0.077474
        3       50      0.05 0.078100
        3       30      0.05 0.079601

Meilleure combinaison :
  maxDepth     : 5.0
  maxIter      : 50.0
  stepSize     : 0.1
  rmse_cv      : 0.07408735528415934


In [30]:
# Évaluation du meilleur modèle sur le test set complet
df_pred_best = cv_model.bestModel.transform(df_test)

rmse_best = evaluator_rmse.evaluate(df_pred_best)
r2_best   = evaluator_r2.evaluate(df_pred_best)
mae_best  = evaluator_mae.evaluate(df_pred_best)

print("Comparaison modèle initial vs meilleur modèle CV :")
print(f"{'Métrique':<8}  {'Initial':>12}  {'CV Best':>12}  {'Gain':>10}")
print("-" * 46)
print(f"{'RMSE':<8}  {rmse:>12.4f}  {rmse_best:>12.4f}  {rmse - rmse_best:>+10.4f}")
print(f"{'MAE':<8}  {mae:>12.4f}  {mae_best:>12.4f}  {mae - mae_best:>+10.4f}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_best:>12.4f}  {r2_best - r2:>+10.4f}")

Comparaison modèle initial vs meilleur modèle CV :
Métrique       Initial       CV Best        Gain
----------------------------------------------
RMSE            0.1011        0.1011     -0.0000
MAE             0.0600        0.0599     +0.0000
R²              0.8045        0.8044     -0.0001


# 1.7 Suivi des expériences avec MLflow

MLflow est un outil de tracking d'expériences ML : il enregistre
hyperparamètres, métriques, artefacts (modèles, graphiques) et permet
de comparer les runs dans une interface web.

```bash
# Pour ouvrir l'interface MLflow dans un terminal séparé :
mlflow ui --backend-store-uri data/output/mlruns --port 5000
# Puis ouvrir http://localhost:5000 dans le navigateur
```


In [32]:
import mlflow
import mlflow.spark

# Configuration du répertoire de tracking
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR.resolve()}")
mlflow.set_experiment("ClimaCity-Paris-GBT")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Expérience          : ClimaCity-Paris-GBT")

2026/09/21 18:17:10 INFO mlflow.tracking.fluent: Experiment with name 'ClimaCity-Paris-GBT' does not exist. Creating a new experiment.


MLflow tracking URI : file:///Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/data/output/mlruns
Expérience          : ClimaCity-Paris-GBT


In [33]:
def evaluer_et_logger(
    df_train_fit, df_test_eval,
    max_depth: int, max_iter: int, step_size: float,
    nom_run: str
) -> dict:
    """Entraîne un pipeline GBT, l'évalue et logue les résultats dans MLflow.

    Args:
        df_train_fit  : DataFrame d'entraînement.
        df_test_eval  : DataFrame de test.
        max_depth     : Profondeur maximale des arbres.
        max_iter      : Nombre d'arbres (iterations).
        step_size     : Taux d'apprentissage.
        nom_run       : Nom du run MLflow.

    Returns:
        Dictionnaire avec les métriques calculées sur le test set.
    """
    with mlflow.start_run(run_name=nom_run):
        # ── Paramètres ──
        mlflow.log_params({
            "max_depth" : max_depth,
            "max_iter"  : max_iter,
            "step_size" : step_size,
            "n_features": len(FEATURES),
            "n_train"   : df_train_fit.count(),
            "n_test"    : df_test_eval.count(),
        })

        # ── Entraînement ──
        gbt_run = GBTRegressor(
            featuresCol="features", labelCol="cible",
            predictionCol="prediction",
            maxIter=max_iter, maxDepth=max_depth,
            stepSize=step_size, seed=SEED
        )
        pipeline_run = Pipeline(stages=[assembler, scaler, gbt_run])
        t0    = time.perf_counter()
        model = pipeline_run.fit(df_train_fit)
        t_fit = time.perf_counter() - t0

        # ── Métriques ──
        preds = model.transform(df_test_eval)
        metriques = {
            "rmse"     : evaluator_rmse.evaluate(preds),
            "mae"      : evaluator_mae.evaluate(preds),
            "r2"       : evaluator_r2.evaluate(preds),
            "fit_time" : round(t_fit, 2),
        }
        mlflow.log_metrics(metriques)

        # ── Modèle ──
        mlflow.spark.log_model(model, artifact_path="pipeline_gbt")

        print(f"  [{nom_run}]  RMSE={metriques['rmse']:.4f}  "
              f"R²={metriques['r2']:.4f}  ({t_fit:.1f}s)")
        return metriques

# ── Plusieurs runs pour comparer ──────────────────────────────────────────────
print("Lancement des runs MLflow...")
configs = [
    {"max_depth": 3, "max_iter": 30, "step_size": 0.1,  "nom_run": "shallow-fast"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.1,  "nom_run": "medium-balanced"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.05, "nom_run": "medium-slow-lr"},
    {"max_depth": 7, "max_iter": 80, "step_size": 0.05, "nom_run": "deep-thorough"},
]

resultats_mlflow = []
for cfg in configs:
    m = evaluer_et_logger(df_train, df_test, **cfg)
    resultats_mlflow.append({"run": cfg["nom_run"], **m})

print("\nTableau comparatif :")
df_res = pd.DataFrame(resultats_mlflow).sort_values("rmse")
print(df_res.to_string(index=False))

Lancement des runs MLflow...


2026/09/21 18:18:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [shallow-fast]  RMSE=0.1044  R²=0.7911  (26.6s)


2026/09/21 18:19:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [medium-balanced]  RMSE=0.1011  R²=0.8045  (52.4s)


2026/09/21 18:20:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [medium-slow-lr]  RMSE=0.1019  R²=0.8010  (53.4s)


26/09/21 18:21:18 WARN DAGScheduler: Broadcasting large task binary with size 1002.6 KiB
26/09/21 18:21:18 WARN DAGScheduler: Broadcasting large task binary with size 1003.1 KiB
26/09/21 18:21:18 WARN DAGScheduler: Broadcasting large task binary with size 1004.1 KiB
26/09/21 18:21:18 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
26/09/21 18:21:19 WARN DAGScheduler: Broadcasting large task binary with size 1007.2 KiB
26/09/21 18:21:19 WARN DAGScheduler: Broadcasting large task binary with size 1011.9 KiB
26/09/21 18:21:19 WARN DAGScheduler: Broadcasting large task binary with size 1021.3 KiB
26/09/21 18:21:19 WARN DAGScheduler: Broadcasting large task binary with size 1025.5 KiB
26/09/21 18:21:19 WARN DAGScheduler: Broadcasting large task binary with size 1026.0 KiB
26/09/21 18:21:20 WARN DAGScheduler: Broadcasting large task binary with size 1027.0 KiB
26/09/21 18:21:20 WARN DAGScheduler: Broadcasting large task binary with size 1027.7 KiB
26/09/21 18:21:20 WAR

  [deep-thorough]  RMSE=0.1006  R²=0.8063  (95.8s)

Tableau comparatif :
            run     rmse      mae       r2  fit_time
  deep-thorough 0.100587 0.059532 0.806288     95.76
medium-balanced 0.101054 0.059959 0.804483     52.39
 medium-slow-lr 0.101948 0.060444 0.801010     53.44
   shallow-fast 0.104449 0.062904 0.791126     26.64


In [35]:
# ── Chargement du meilleur modèle depuis MLflow ───────────────────────────────
from mlflow.tracking import MlflowClient

client  = MlflowClient()
exp     = client.get_experiment_by_name("ClimaCity-Paris-GBT")
runs    = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"],
    max_results=1
)
best_run = runs[0]

print(f"Meilleur run :")
print(f"  ID   : {best_run.info.run_id}")
print(f"  Nom  : {best_run.data.tags.get('mlflow.runName', 'N/A')}")
print(f"  RMSE : {best_run.data.metrics['rmse']:.4f}")
print(f"  R²   : {best_run.data.metrics['r2']:.4f}")

# Rechargement du modèle
model_uri     = f"runs:/{best_run.info.run_id}/pipeline_gbt"
model_recharge = mlflow.spark.load_model(model_uri)
print(f"\nModèle rechargé depuis : {model_uri}")

# Vérification : les prédictions sont identiques
df_verif = model_recharge.transform(df_test.limit(100))
rmse_verif = evaluator_rmse.evaluate(df_verif)
print(f"RMSE sur 100 lignes (vérification) : {rmse_verif:.4f}")

Meilleur run :
  ID   : e188133739314205be7ac5391ed620e1
  Nom  : deep-thorough
  RMSE : 0.1006
  R²   : 0.8063


2026/09/21 18:22:42 INFO mlflow.spark: URI 'runs:/e188133739314205be7ac5391ed620e1/pipeline_gbt/sparkml' does not point to the current DFS.
2026/09/21 18:22:42 INFO mlflow.spark: File 'runs:/e188133739314205be7ac5391ed620e1/pipeline_gbt/sparkml' not found on DFS. Will attempt to upload the file.



Modèle rechargé depuis : runs:/e188133739314205be7ac5391ed620e1/pipeline_gbt
RMSE sur 100 lignes (vérification) : 0.0281


In [37]:
# Sauvegarde locale du meilleur modèle (format natif Spark)
chemin_model_local = MODELS_DIR / "gbt_best"
cv_model.bestModel.write().overwrite().save(str(chemin_model_local))
print(f"Modèle sauvegardé localement : {chemin_model_local}")

# Pour le recharger plus tard :
# from pyspark.ml import PipelineModel
# model_recharge = PipelineModel.load(str(chemin_model_local))

Modèle sauvegardé localement : data/output/models/gbt_best


26/09/22 09:22:21 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:307)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1937